# TabICL v1 on all 200 TALENT classification datasets
Select a GPU runtime and run all cells. Allow Google Drive access to preserve results between sessions. The workbook updates after every dataset. Run one copy of the notebook at a time per Drive output folder.

The checkpoint uses labeled training rows as context and predicts held-out test labels. The 32 ensemble members vary preprocessing, column order and class order. Validation rows are unused. All 120 binary and 80 multiclass tasks are included, extending beyond the paper's main 171-dataset comparison.

In [ ]:
import subprocess, sys
COMMIT = '7ccaddaa45f9d586d90df37126d39a0cb2a964e7'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/soda-inria/tabicl.git@'+COMMIT, 'openpyxl'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, io, json, zipfile, time, gc, traceback
import numpy as np
import pandas as pd
import sklearn
from tabicl import TabICLClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

drive.mount('/content/drive')
output = Path('/content/drive/MyDrive/TabICL_TALENT_v1_200')
scores = output / 'results'
scores.mkdir(parents=True, exist_ok=True)
work = Path('/content/tabicl_benchmark')
work.mkdir(exist_ok=True)
checkpoint = 'tabicl-classifier-v1-20250208.ckpt'
def fetch(name, url, digest):
    path = work / name
    def sha(p):
        h = hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda: f.read(2**20), b''):
                h.update(block)
        return h.hexdigest()
    if path.exists() and sha(path) == digest:
        return path
    tmp = path.with_suffix('.part')
    subprocess.run(['curl','-fL','--retry','3',url,'-o',str(tmp)],check=True)
    assert sha(tmp) == digest, f'Unexpected download for {name}'
    tmp.replace(path)
    return path
weights = fetch(checkpoint, 'https://huggingface.co/jingang/TabICL/resolve/main/'+checkpoint, '04c5c1d261c1f782dc9b263990dcfa5152b1949ed8451ad16c010cdafbea07e0')
archive = fetch('dataset-latest.zip','https://huggingface.co/datasets/LAMDA-Tabular/TALENT/resolve/main/dataset-latest.zip','4c8481107153593eb98ef5bb677b00dedfa04cd4c31441babf696a8d100f207c')
z = zipfile.ZipFile(archive)
tasks = {}
for path in z.namelist():
    if path.endswith('/info.json'):
        meta = json.loads(z.read(path))
        if meta['task_type'] in ('binclass','multiclass'):
            tasks[path.split('/')[-2]] = meta
from collections import Counter
assert Counter(d['task_type'] for d in tasks.values()) == {'binclass':120,'multiclass':80}
print('Verified both downloads and all 200 classification datasets.')

In [ ]:
def array(name, prefix, split):
    return np.load(io.BytesIO(z.read(f'data/{name}/{prefix}_{split}.npy')), allow_pickle=True)

def features(name, split):
    parts = []
    for prefix in ['N','C']:
        if f'data/{name}/{prefix}_{split}.npy' in z.NameToInfo:
            df = pd.DataFrame(array(name,prefix,split))
            if prefix == 'N':
                df = df.apply(pd.to_numeric, errors='raise').astype(float)
            else:
                df = df.astype('string').astype(object).where(df.notna(),np.nan)
            parts.append(df.add_prefix(prefix))
    return pd.concat(parts,axis=1)

def evaluate(name):
    x, xt = features(name,'train'), features(name,'test')
    y = array(name,'y','train').ravel()
    yt = array(name,'y','test').ravel()
    clf = TabICLClassifier(model_path=str(weights),checkpoint_version=checkpoint,
            n_estimators=32, batch_size=1, device='cuda', use_amp=False,
            allow_auto_download=False, random_state=42)
    start = time.monotonic()
    clf.fit(x,y)
    p = clf.predict_proba(xt)
    elapsed = time.monotonic()-start
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1,atol=1e-5)
    pred = clf.classes_[p.argmax(axis=1)]
    # AUC requires both positive and negative examples for each scored class.
    aucs = [roc_auc_score(yt==c,p[:,j]) for j,c in enumerate(clf.classes_)
            if 0 < (yt==c).sum() < len(yt)]
    return {'Status':'Scored','Train':len(y),'Test':len(yt),'Classes':len(clf.classes_),
            'Accuracy':float(accuracy_score(yt,pred)), 'ROC AUC':float(np.mean(aucs)) if aucs else None,
            'AUC classes':len(aucs),'Log loss':float(log_loss(yt,p,labels=clf.classes_)),
            'Correct':int((pred==yt).sum()),'Seconds':elapsed,
            'GPU':torch.cuda.get_device_name(0),'PyTorch':str(torch.__version__),'sklearn':sklearn.__version__}

def record(name):
    path = scores / (name+'.json')
    return json.loads(path.read_text()) if path.exists() else {'Status':'Pending'}

def save(name,result):
    dest = scores/(name+'.json')
    temp = dest.with_suffix('.tmp')
    temp.write_text(json.dumps(result,allow_nan=False))
    temp.replace(dest)

def export():
    rows = [{'Dataset':n,'Task':m['task_type'],**record(n)} for n,m in sorted(tasks.items())]
    frame = pd.DataFrame(rows)
    done = frame['Status'].eq('Scored').sum()
    metadata = [('Scored datasets',f'{done} / 200'),('Code commit',COMMIT),('Model',checkpoint),
        ('Data','LAMDA-Tabular/TALENT • dataset-latest.zip • official partitions'),
        ('Settings','32 members, seed 42, AMP off, full train and test partitions, no validation tuning'),
        ('ROC AUC','Macro one-vs-rest across test-observed classes with positives and negatives.'),
        ('Accuracy','Fraction of correct predictions. Higher is better.'),
        ('Log loss','Mean negative log probability of the true label. Lower is better.'),
        ('Seconds','Fit and probability prediction, excluding data loading and metric calculation.')]
    tmp = output/'TALENT_200.tmp.xlsx'
    with pd.ExcelWriter(tmp,engine='openpyxl') as writer:
        pd.DataFrame(metadata,columns=['Setting','Value']).to_excel(writer,sheet_name='Run',index=False)
        frame.to_excel(writer,sheet_name='Dataset metrics',index=False)
        from openpyxl.utils import get_column_letter
        for ws in writer.book.worksheets:
            ws.freeze_panes='A2'
            ws.auto_filter.ref=ws.dimensions
            for j,col in enumerate(ws.columns,1):
                ws.column_dimensions[get_column_letter(j)].width=min(110,max(13,max(len(str(c.value or '')) for c in col)+2))
    tmp.replace(output/'TALENT_200.xlsx')
    return int(done)

## Run the benchmark
Smaller datasets run first. Each completed dataset is saved to Drive before the next one starts. A failed dataset gets an error message in Excel and is retried the next time you run this cell. If memory runs out on a large dataset, reconnect with a higher-memory GPU runtime and run all cells again.

In [ ]:
order = sorted(tasks,key=lambda n:tasks[n].get('train_size',0)*(10+tasks[n].get('n_num_features',0)+tasks[n].get('n_cat_features',0)))
for name in order:
    if record(name)['Status']=='Scored':
        continue
    print('Running',name,flush=True)
    try:
        result = evaluate(name)
    except Exception as e:
        traceback.print_exc()
        result = {'Status':'Failed','Error':f'{type(e).__name__}: {e}'}
    save(name,result)
    gc.collect()
    torch.cuda.empty_cache()
    count = export()
    print(f"{count}/200 scored • {name} • {result['Status']}",flush=True)
print(f'{export()}/200 scored. Workbook: {output / "TALENT_200.xlsx"}')